# Use Python function to predict user's satisfaction for car rental company with `ibm-watsonx-ai`

This notebook trains a **Keras** (TensorFlow) model to predict customer satisfaction based on the feedback that has been provided. The notebook also demonstrates how you can use the **python function** for the deep learning model data preprocessing required before you start model scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.

Contents:
1. [Set up the environment](#1.-Set-up-the-environment)
2. [Load and explore data](#2.-Load-and-explore-data)
3. [Create Keras model using TensorFlow backend](#3.-Create-a-Keras-model-using-the-TensorFlow-backend)
4. [Store the model in the repository](#4.-Store-the-model-in-the-repository)
5. [Deploy the model](#5.-Deploy-the-model)
6. [Score the model](#6.-Score-the-model)
7. [Define, store and deploy Python function](#7.-Define,-store-and-deploy-Python-function)
8. [Clean up](#8.-Clean-up)
9. [Summary and next steps](#9.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import the `ibm-watsonx-ai` and dependecies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install "scikit-learn>=1.6,<1.7" | tail -n 1
%pip install "tensorflow>=2.18,<2.19" | tail -n 1
%pip install "keras>=3.12,<3.13" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

**Note:** If you're encountering `ModuleNotFoundError: No module named 'distutils'` exception, install the `setuptools` package and try again.

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click New Deployment Space
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press Create
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [ ]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Load-and-explore-data"></a>
## 2. Load and explore data

In this section you read in the `car_rental_training_data.csv` file, load it as a pandas dataFrame and then perform a basic exploration. 


In [6]:
import wget

link_to_data = "https://github.com/IBM/watsonx-ai-samples/raw/master/cloud/data/cars-4-you/car_rental_training_data.csv"
filename = wget.download(link_to_data)

Load the data as a pandas dataFrame.


In [7]:
import pandas as pd

data = pd.read_csv(filename, sep=";")
data.head()

,ID,Gender,Status,Children,Age,Customer_Status,Car_Owner,Customer_Service,Satisfaction,Business_Area,Action
0,83,Female,M,2,48.85,Inactive,Yes,I thought the representative handled the initi...,0,Product: Availability/Variety/Size,Free Upgrade
1,1307,Female,M,0,55.00,Inactive,No,I have had a few recent rentals that have take...,0,Product: Availability/Variety/Size,Voucher
2,1737,Male,M,0,42.35,Inactive,Yes,car cost more because I didn't pay when I rese...,0,Product: Availability/Variety/Size,Free Upgrade
3,3721,Male,M,2,61.71,Inactive,Yes,I didn't get the car I was told would be avail...,0,Product: Availability/Variety/Size,Free Upgrade
4,11,Male,S,2,56.47,Active,No,If there was not a desired vehicle available t...,1,Product: Availability/Variety/Size,NaN




**Note:** 0 - not satisfied, 1 - satisfied


Extract the required columns and count the number of records.


In [8]:
complain_data = data[["Customer_Service", "Satisfaction"]]

In [9]:
complain_data.count()

Customer_Service    486
Satisfaction        486
dtype: int64

<a id="3.-Create-a-Keras-model-using-the-TensorFlow-backend"></a>
## 3. Create a Keras model using the TensorFlow backend

### 3.1 Prepare the data


In [10]:
from keras.layers import TextVectorization

max_features = 500

for idx, row in complain_data.iterrows():
    row[0] = row[0].replace("rt", " ")

tokenizer = TextVectorization(
    output_sequence_length=100,
    max_tokens=max_features,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    split="whitespace",
    pad_to_max_tokens=True,
)
tokenizer.adapt(complain_data["Customer_Service"].values)

X = tokenizer(complain_data["Customer_Service"].values)
X

<tf.Tensor: shape=(486, 100), dtype=int64, numpy=
array([[  5, 206,   2, ...,   0,   0,   0],
       [  5,  20,  25, ...,   0,   0,   0],
       [  7, 191,  58, ...,   0,   0,   0],
       ...,
       [ 23,  73,   2, ...,   0,   0,   0],
       [  5,  19,  10, ...,   0,   0,   0],
       [ 21,   1,   5, ...,   0,   0,   0]])>

### Split the data into train and test data sets.


In [11]:
from sklearn.model_selection import train_test_split

Y = complain_data["Satisfaction"].values
X_train, X_test, Y_train, Y_test = train_test_split(
    X.numpy(), Y, test_size=0.33, random_state=42
)

X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

((325, 100), (325,), (161, 100), (161,))

### 3.2 Design and train the model
Create the network definition based on the Gated Recurrent Unit (Cho et al. 2014).


In [12]:
from keras.layers import LSTM, Conv1D, Dense, Embedding, MaxPooling1D
from keras.models import Sequential

embedding_vector_length = 100

model = Sequential()
model.add(Embedding(max_features, embedding_vector_length))
model.add(Conv1D(filters=32, kernel_size=3, padding="same", activation="relu"))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(100))
model.add(Dense(1, activation="sigmoid"))
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train the model.


In [14]:
history = model.fit(
    X_train, Y_train, validation_data=(X_test, Y_test), epochs=50, batch_size=64
)

In [15]:
import numpy

"Best accuracy on test: %3.3f" % numpy.array(history.history["val_accuracy"]).max()

'Best accuracy on test: 0.901'

**Note:** For purpose of this demo, model tuning has been skipped.

Store and archive the model in the notebook filesystem.



### Evaluate the model

In [16]:
scores = model.evaluate(X_test, Y_test)
"Evaluation Accuracy: %.2f%%" % (scores[1] * 100)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8137 - loss: 0.4733 


'Evaluation Accuracy: 81.37%'

In [17]:
import os

filename = "complain_model.h5"
model.save(filename)

# Compress Keras model
tar_filename = filename + ".tgz"
cmdstring = "tar -zcvf " + tar_filename + " " + filename
score = tar_filename
os.system(cmdstring)

0

<a id="4.-Store-the-model-in-the-repository"></a>
## 4. Store the model in the repository

In this section, you will learn how to store your model in watsonx.ai Runtime repository by using the watsonx.ai Client.

### 4.1: Publish model
#### Publish model in watsonx.ai Runtime repository on Cloud.


Get software specification for tensorflow model.

In [18]:
software_spec_id = client.software_specifications.get_id_by_name("runtime-25.1-py3.12")
print(software_spec_id)

f47ae1c3-198e-5718-b59d-2ea471561e9e


Store model

In [19]:
metadata = {
    client.repository.ModelMetaNames.NAME: "CARS4U - Satisfaction Prediction Model",
    client.repository.ModelMetaNames.TYPE: "tensorflow_2.18",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: software_spec_id,
}
print(metadata)
stored_model_details = client.repository.store_model(tar_filename, meta_props=metadata)

{'name': 'CARS4U - Satisfaction Prediction Model', 'type': 'tensorflow_2.18', 'software_spec': 'f47ae1c3-198e-5718-b59d-2ea471561e9e'}


### 4.2: Get model details

In [20]:
import json

model_id = client.repository.get_model_id(stored_model_details)

model_details = client.repository.get_details(model_id)
print(json.dumps(model_details, indent=2))

{
  "metadata": {
    "name": "CARS4U - Satisfaction Prediction Model",
    "space_id": "fb3d528a-bf16-460e-bcf6-06f05d8ba57c",
    "resource_key": "bbcab539-4ae3-449a-81ee-c744ddc15f50",
    "id": "e23efabc-1daf-4517-9e72-5d3f431941eb",
    "created_at": "2026-01-16T13:17:58Z",
    "rov": {
      "member_roles": {
        "IBMid-696000GJGB": {
          "user_iam_id": "IBMid-696000GJGB",
          "roles": [
            "OWNER"
          ]
        }
      }
    },
    "owner": "IBMid-696000GJGB"
  },
  "entity": {
    "software_spec": {
      "id": "f47ae1c3-198e-5718-b59d-2ea471561e9e"
    },
    "type": "tensorflow_2.18"
  }
}


<a id="5.-Deploy-the-model"></a>
## 5. Deploy the model
In this section you will learn how to create batch deployment to create job using the watsonx.ai Client.

You can use commands below to create batch deployment for stored model (web service).

### 5.1: Create model deployment


In [21]:
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "CARS4U - Satisfaction Prediction Model Deployment",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

In [22]:
deployment_details = client.deployments.create(
    model_id, meta_props, background_mode=False
)



######################################################################################

Synchronous deployment creation for id: 'e23efabc-1daf-4517-9e72-5d3f431941eb' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='0927c2e2-af86-4e32-ad8e-eee462457ead'
-----------------------------------------------------------------------------------------------




**Note**: Here we use deployment url saved in published_model object. In next section, we show how to retrieve deployment url from watsonx.ai Runtime instance.



In [23]:
deployment_id = client.deployments.get_id(deployment_details)

Now, You can list all deployments.

In [24]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,0927c2e2-af86-4e32-ad8e-eee462457ead,CARS4U - Satisfaction Prediction Model Deployment,ready,2026-01-16T13:18:08.220Z,model,supported,
1,3f9aae63-6c4b-4f9f-bc52-7f091a983625,CARS4U - Satisfaction Prediction - AI Function...,ready,2026-01-16T13:12:44.294Z,function,supported,
2,b8eef947-b94c-4bcd-9502-d97de27ddfd4,CARS4U - Satisfaction Prediction Model Deployment,ready,2026-01-16T12:42:08.485Z,model,supported,


### 5.2 Get deployment details

In [25]:
client.deployments.get_details(deployment_id)

<a id="6.-Score-the-model"></a>
## 6. Score the model
Let's see if our deployment works.


You can use below method to do test scoring request against deployed model.



In [26]:
deployment_id = client.deployments.get_id(deployment_details)

In [27]:
deployment_id

'0927c2e2-af86-4e32-ad8e-eee462457ead'

**Action**: Prepare scoring payload with records to score.

In [28]:
index = 5

scoring_data = X[index]
print(X_test[index])
print(Y_test[index])

[  5  39  20   4 209  26 242   7  24 349   2 255  13  11   9 110  15  33
   6  55   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0]
1


In [29]:
scoring_payload = {
    client.deployments.ScoringMetaNames.INPUT_DATA: [{"values": [scoring_data.numpy()]}]
}

In [30]:
scores = client.deployments.score(deployment_id, scoring_payload)

Let's print the scoring results.

In [31]:
print(str(scores))

{'predictions': [{'fields': ['prediction', 'prediction_classes', 'probability'], 'id': 'keras_tensor_14', 'values': [[[0.21466058492660522], [0], [0.21466058492660522]]]}]}


<a id="7.-Define,-store-and-deploy-Python-function"></a>
## 7. Define, store and deploy Python function
Let's define a function that does data preprocessing and model scoring. As we saw in the previous cells, the model expects numerical input, so the text comment needs to be preprocessed.

### 7.1 Define generic parameters

In [32]:
parameters = {
    "deployment_id": deployment_id,
    "credentials": credentials.to_dict(),
    "vocabulary": list(map(str, tokenizer.get_vocabulary())),
    "space_id": space_id,
}

In [33]:
def score_generator(params=parameters):
    import re

    from ibm_watsonx_ai import APIClient

    client = APIClient(params["credentials"])
    client.set.default_space(params["space_id"])

    def score(payload):
        maxlen = 100
        preprocessed_records = []
        complain_data = payload["input_data"][0]["values"]

        for data in complain_data:
            comment = data[0]
            cleanString = re.sub(r"[!\"#$%&()*+,-./:;<=>?@[\]^_`{|}~]", "", comment)
            splitted_comment = cleanString.split()[:maxlen]
            hashed_tokens = []

            for token in splitted_comment:
                if token not in params["vocabulary"]:
                    continue

                index = params["vocabulary"].index(token)
                if index < 501:
                    hashed_tokens.append(index)

            hashed_tokens_size = len(hashed_tokens)
            padded_tokens = [0] * (maxlen - hashed_tokens_size) + hashed_tokens
            preprocessed_records.append(padded_tokens)

        scoring_payload = {"values": preprocessed_records}
        scoring_payload = {
            client.deployments.ScoringMetaNames.INPUT_DATA: [scoring_payload]
        }

        return client.deployments.score(params["deployment_id"], scoring_payload)

    return score

#### Test the function locally


In [34]:
sample_payload = {
    "input_data": [
        {
            "fields": ["feedback"],
            "values": [
                [
                    "delayed shuttle, almost missed flight, bad customer service",
                ],
                [
                    "The car was great and they were able to provide all features I wanted with limited time they had.",
                ],
            ],
        }
    ]
}

In [35]:
print(sample_payload)

{'input_data': [{'fields': ['feedback'], 'values': [['delayed shuttle, almost missed flight, bad customer service'], ['The car was great and they were able to provide all features I wanted with limited time they had.']]}]}


In [36]:
score = score_generator()
score_ai = score(sample_payload)
print(score_ai)

{'predictions': [{'fields': ['prediction', 'prediction_classes', 'probability'], 'id': 'keras_tensor_14', 'values': [[[0.9685041308403015], [1], [0.9685041308403015]], [[0.9675772190093994], [1], [0.9675772190093994]]]}]}


**Note:** 0 - not satisfied. 1 - satisfied

### 7.2 Store the function

In [37]:
software_spec_id = client.software_specifications.get_id_by_name("runtime-25.1-py3.12")
software_spec_id

'f47ae1c3-198e-5718-b59d-2ea471561e9e'

In [38]:
meta_data = {
    client.repository.FunctionMetaNames.NAME: "CARS4U - Satisfaction Prediction - AI Function",
    client.repository.FunctionMetaNames.SOFTWARE_SPEC_ID: software_spec_id,
}

function_details = client.repository.store_function(score_generator, meta_data)

In [39]:
client.repository.list_functions()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,08735666-6568-40ff-bbde-697802b976c9,CARS4U - Satisfaction Prediction - AI Function,2026-01-16T13:19:33Z,python,supported,


### 7.3 Deploy the function

In [40]:
function_id = client.repository.get_function_id(function_details)
published_function_id = client.repository.get_function_id(function_details)
client.repository.get_function_details(function_id)

{'metadata': {'name': 'CARS4U - Satisfaction Prediction - AI Function',
  'space_id': 'fb3d528a-bf16-460e-bcf6-06f05d8ba57c',
  'resource_key': 'f213e8fa-c396-46ca-818c-cd484942f949',
  'id': '08735666-6568-40ff-bbde-697802b976c9',
  'created_at': '2026-01-16T13:19:33Z',
  'rov': {'member_roles': {'IBMid-696000GJGB': {'user_iam_id': 'IBMid-696000GJGB',
     'roles': ['OWNER']}}},
  'owner': 'IBMid-696000GJGB'},
 'entity': {'software_spec': {'id': 'f47ae1c3-198e-5718-b59d-2ea471561e9e'},
  'type': 'python'}}

In [41]:
meta_data = {
    client.deployments.ConfigurationMetaNames.NAME: "CARS4U - Satisfaction Prediction - AI Function Deployment",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

In [42]:
function_deployment_details = client.deployments.create(
    published_function_id, meta_data
)



######################################################################################

Synchronous deployment creation for id: '08735666-6568-40ff-bbde-697802b976c9' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='408bf067-2b38-4e26-96fe-47f9f2d0bf7d'
-----------------------------------------------------------------------------------------------




### 7.4 Score function

In [43]:
import json

print(function_deployment_details["metadata"]["id"])
scoring_response = client.deployments.score(
    function_deployment_details["metadata"]["id"], sample_payload
)

print(json.dumps(scoring_response, indent=3))

408bf067-2b38-4e26-96fe-47f9f2d0bf7d
{
   "predictions": [
      {
         "fields": [
            "prediction",
            "prediction_classes",
            "probability"
         ],
         "id": "keras_tensor_14",
         "values": [
            [
               [
                  0.9685041308403015
               ],
               [
                  1
               ],
               [
                  0.9685041308403015
               ]
            ],
            [
               [
                  0.9675772190093994
               ],
               [
                  1
               ],
               [
                  0.9675772190093994
               ]
            ]
         ]
      }
   ]
}


<a id="8.-Clean-up"></a>
## 8. Clean up

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="9.-Summary-and-next-steps"></a>
## 9. Summary and next steps

You successfully completed this notebook! You learned how to use Keras machine learning library as well as watsonx.ai Runtime for model creation and deployment. Check out our _[Online Documentation](https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Wojciech Jargielo**, Software Engineer.

**Rafał Chrzanowski**, Software Engineer at watsonx.ai.


Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.